# 06 - Neural Network Architecture
Visualize, analyze, and iterate on the DrivePolicy architecture. Covers model summary, per-encoder breakdown, forward pass shape tracing, weight distributions, and architecture comparison.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
from torchinfo import summary
from pufferlib.ocean.drive.drive import Drive
from pufferlib.ocean.drive import binding
from pufferlib.ocean.torch import Drive as DrivePolicy, DriveBackbone, Recurrent

# --- Environment configuration ---
NUM_AGENTS = 64
SIMULATION_MODE = "gigaflow"
DYNAMICS_MODEL = "jerk"
ACTION_TYPE = "discrete"
DT = 0.1
SCENARIO_LENGTH = 512
RESAMPLE_FREQUENCY = 0
REWARD_CONDITIONING = True
REWARD_RANDOMIZATION = False
TARGET_TYPE = "static"
COLLISION_BEHAVIOR = 1
OFFROAD_BEHAVIOR = 1
SEED = 42
MAP_DIR = "../pufferlib/resources/drive/binaries/carla"

# --- Observation dimensions ---
MAX_PARTNERS = 20
MAX_LANES = 100
MAX_BOUNDS = 50
MAX_TRAFFIC = 4
MAX_STOP_SIGNS = 0

# --- Policy architecture ---
INPUT_SIZE = 64
BACKBONE_HIDDEN_SIZE = 1024
BACKBONE_NUM_LAYERS = 3
ACTOR_HIDDEN_SIZE = 128
ACTOR_NUM_LAYERS = 3
CRITIC_HIDDEN_SIZE = 64
CRITIC_NUM_LAYERS = 2
SPLIT_NETWORK = False
ENCODER_GIGAFLOW = True
DROPOUT = 0.0

# --- Derived from binding ---
EGO_DIM = binding.EGO_FEATURES_JERK
NUM_COEFS = binding.NUM_REWARD_COEFS
PARTNER_F = binding.PARTNER_FEATURES
ROAD_F = binding.ROAD_FEATURES
TRAFFIC_CONTROL_F = binding.TRAFFIC_CONTROL_FEATURES
NUM_TRAFFIC_CONTROL_TYPES = binding.NUM_TRAFFIC_CONTROL_TYPES

# --- Create environment ---
env = Drive(
    num_agents=NUM_AGENTS,
    num_maps=1,
    min_agents_per_env=NUM_AGENTS,
    max_agents_per_env=NUM_AGENTS,
    simulation_mode=SIMULATION_MODE,
    dynamics_model=DYNAMICS_MODEL,
    action_type=ACTION_TYPE,
    dt=DT,
    scenario_length=SCENARIO_LENGTH,
    resample_frequency=RESAMPLE_FREQUENCY,
    reward_conditioning=REWARD_CONDITIONING,
    reward_randomization=REWARD_RANDOMIZATION,
    target_type=TARGET_TYPE,
    map_dir=MAP_DIR,
    collision_behavior=COLLISION_BEHAVIOR,
    offroad_behavior=OFFROAD_BEHAVIOR,
    obs_slots_lane=MAX_LANES,
    obs_slots_boundary=MAX_BOUNDS,
    obs_slots_partners=MAX_PARTNERS,
    seed=SEED,
)
obs, info = env.reset(seed=SEED)

MAX_TARGET = env.num_target_waypoints
TARGET_F = binding.STATIC_TARGET_FEATURES if TARGET_TYPE == "static" else binding.DYNAMIC_TARGET_FEATURES
TARGET_DIM = MAX_TARGET * TARGET_F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
policy = DrivePolicy(
    env,
    input_size=INPUT_SIZE,
    backbone_hidden_size=BACKBONE_HIDDEN_SIZE,
    backbone_num_layers=BACKBONE_NUM_LAYERS,
    actor_hidden_size=ACTOR_HIDDEN_SIZE,
    actor_num_layers=ACTOR_NUM_LAYERS,
    critic_hidden_size=CRITIC_HIDDEN_SIZE,
    critic_num_layers=CRITIC_NUM_LAYERS,
    split_network=SPLIT_NETWORK,
    encoder_gigaflow=ENCODER_GIGAFLOW,
    dropout=DROPOUT,
).to(device)

print(f"Device: {device}")
print(f"Obs dim: {obs.shape[1]}")
print(f"Action dim: {policy.atn_dim}")
print(f"Split network: {SPLIT_NETWORK}")
print(f"Backbone: {BACKBONE_HIDDEN_SIZE} x {BACKBONE_NUM_LAYERS}L")
print(f"Actor: {ACTOR_HIDDEN_SIZE} x {ACTOR_NUM_LAYERS}L")
print(f"Critic: {CRITIC_HIDDEN_SIZE} x {CRITIC_NUM_LAYERS}L")
print(f"Encoder gigaflow: {ENCODER_GIGAFLOW}, Dropout: {DROPOUT}")

## Model Summary (torchinfo)

In [ ]:
obs_tensor = torch.FloatTensor(obs).to(device)
summary(policy, input_data=obs_tensor, depth=4, col_names=["input_size", "output_size", "num_params", "mult_adds"])

## Architecture Diagram

In [ ]:
backbone = policy.actor_backbone
cond_dim = backbone.conditioning_dim

# Collect encoder info — encoder_gigaflow adds Tanh+Dropout between LN and second Linear
# ego, partner, conditioning use encoder_gigaflow; lane, boundary, traffic_ctrl use dropout
encoders = [
    ("ego", EGO_DIM, 1, "direct", ENCODER_GIGAFLOW),
    ("conditioning", cond_dim, 1, "direct", ENCODER_GIGAFLOW) if cond_dim > 0 else None,
    ("partner", PARTNER_F, MAX_PARTNERS, "max-pool", ENCODER_GIGAFLOW),
    ("lane", ROAD_F, MAX_LANES, "max-pool", ENCODER_GIGAFLOW),
    ("boundary", ROAD_F, MAX_BOUNDS, "max-pool", ENCODER_GIGAFLOW),
    (
        "traffic_ctrl",
        TRAFFIC_CONTROL_F - 2 + binding.NUM_TRAFFIC_CONTROL_TYPES + binding.NUM_TRAFFIC_CONTROL_STATES,
        MAX_TRAFFIC,
        "max-pool (onehot)",
        ENCODER_GIGAFLOW,
    ),
]
encoders = [e for e in encoders if e is not None]

fig, ax = plt.subplots(figsize=(14, 8))
ax.set_xlim(0, 10)
ax.set_ylim(0, 10)
ax.axis("off")

n_enc = len(encoders)
y_positions = np.linspace(9, 1, n_enc)
colors = plt.cm.Set2(np.linspace(0, 1, n_enc))

# Draw encoders
for i, ((name, in_f, n_obj, agg, gigaflow), y, c) in enumerate(zip(encoders, y_positions, colors)):
    # Input box
    label = f"{name}\n{n_obj}x{in_f}" if n_obj > 1 else f"{name}\n{in_f}"
    ax.add_patch(plt.Rectangle((0.2, y - 0.3), 1.6, 0.6, facecolor=c, edgecolor="black", lw=1.2, alpha=0.8))
    ax.text(1.0, y, label, ha="center", va="center", fontsize=8, fontweight="bold")

    # Encoder box — show gigaflow arch vs standard
    ax.add_patch(plt.Rectangle((2.5, y - 0.25), 2.0, 0.5, facecolor="lightyellow", edgecolor="black", lw=1))
    ax.text(3.5, y + 0.05, f"Linear({in_f},{INPUT_SIZE})", ha="center", va="center", fontsize=7)
    if gigaflow:
        ax.text(
            3.5,
            y - 0.12,
            f"LN+Tanh+Drop+Linear({INPUT_SIZE},{INPUT_SIZE})",
            ha="center",
            va="center",
            fontsize=5.5,
            color="darkgreen",
        )
    else:
        drop_str = f"+Drop({DROPOUT})" if DROPOUT > 0 and name not in ("ego", "partner", "conditioning") else ""
        ax.text(
            3.5,
            y - 0.12,
            f"LN{drop_str}+Linear({INPUT_SIZE},{INPUT_SIZE})",
            ha="center",
            va="center",
            fontsize=6,
            color="gray",
        )

    # Aggregation
    if n_obj > 1:
        ax.text(5.0, y, agg, ha="center", va="center", fontsize=7, style="italic", color="darkblue")
        arrow_start = 5.5
    else:
        arrow_start = 4.6

    # Arrows
    ax.annotate("", xy=(2.5, y), xytext=(1.8, y), arrowprops=dict(arrowstyle="->", lw=1))
    ax.annotate("", xy=(6.0, 5.0), xytext=(arrow_start, y), arrowprops=dict(arrowstyle="->", lw=0.8, color="gray"))

# Concat box
ax.add_patch(plt.Rectangle((5.8, 4.5), 1.4, 1.0, facecolor="lightsalmon", edgecolor="black", lw=1.5))
ax.text(6.5, 5.2, "Concat", ha="center", va="center", fontsize=9, fontweight="bold")
ax.text(6.5, 4.85, f"{n_enc}x{INPUT_SIZE}={n_enc * INPUT_SIZE}", ha="center", va="center", fontsize=7)

# Backbone
ax.add_patch(plt.Rectangle((7.5, 4.5), 1.3, 1.0, facecolor="lightblue", edgecolor="black", lw=1.5))
ax.text(8.15, 5.15, f"Backbone ({BACKBONE_NUM_LAYERS}L)", ha="center", va="center", fontsize=8, fontweight="bold")
ax.text(8.15, 4.85, f"GELU+Linear\n({n_enc * INPUT_SIZE},{BACKBONE_HIDDEN_SIZE})", ha="center", va="center", fontsize=6)
ax.annotate("", xy=(7.5, 5.0), xytext=(7.2, 5.0), arrowprops=dict(arrowstyle="->", lw=1.5))

# Actor / Critic heads
ax.add_patch(plt.Rectangle((9.0, 5.7), 0.9, 0.6, facecolor="lightgreen", edgecolor="black", lw=1.2))
actor_label = f"Actor ({ACTOR_NUM_LAYERS}L)\n{BACKBONE_HIDDEN_SIZE}->{sum(policy.atn_dim)}"
if ACTOR_NUM_LAYERS > 1:
    actor_label = (
        f"Actor ({ACTOR_NUM_LAYERS}L)\n{BACKBONE_HIDDEN_SIZE}->{ACTOR_HIDDEN_SIZE}->...->{sum(policy.atn_dim)}"
    )
ax.text(9.45, 6.0, actor_label, ha="center", va="center", fontsize=6, fontweight="bold")

ax.add_patch(plt.Rectangle((9.0, 3.7), 0.9, 0.6, facecolor="plum", edgecolor="black", lw=1.2))
critic_label = f"Critic ({CRITIC_NUM_LAYERS}L)\n{BACKBONE_HIDDEN_SIZE}->1"
if CRITIC_NUM_LAYERS > 1:
    critic_label = f"Critic ({CRITIC_NUM_LAYERS}L)\n{BACKBONE_HIDDEN_SIZE}->{CRITIC_HIDDEN_SIZE}->...->1"
ax.text(9.45, 4.0, critic_label, ha="center", va="center", fontsize=6, fontweight="bold")

ax.annotate("", xy=(9.0, 6.0), xytext=(8.8, 5.3), arrowprops=dict(arrowstyle="->", lw=1.2))
ax.annotate("", xy=(9.0, 4.0), xytext=(8.8, 4.7), arrowprops=dict(arrowstyle="->", lw=1.2))

split_label = "SPLIT" if SPLIT_NETWORK else "SHARED"
ax.text(8.9, 4.55, split_label, ha="center", va="center", fontsize=7, color="red", fontweight="bold")

gigaflow_label = "GIGAFLOW" if ENCODER_GIGAFLOW else "STANDARD"
ax.text(
    5.0,
    0.3,
    f"Encoder mode: {gigaflow_label} | Dropout: {DROPOUT}",
    ha="center",
    va="center",
    fontsize=8,
    color="darkgreen",
    fontweight="bold",
)

ax.set_title(
    f"DrivePolicy Architecture (input_size={INPUT_SIZE}, backbone={BACKBONE_HIDDEN_SIZE})",
    fontsize=12,
    fontweight="bold",
)
plt.tight_layout()
plt.show()

## Per-Encoder Parameter Breakdown

In [ ]:
def count_params(module):
    return sum(p.numel() for p in module.parameters())


backbone = policy.actor_backbone
components = {
    "ego_encoder": backbone.ego_encoder,
    "lane_encoder": backbone.lane_encoder,
    "boundary_encoder": backbone.boundary_encoder,
    "partner_encoder": backbone.partner_encoder,
    "traffic_ctrl_encoder": backbone.traffic_control_encoder,
}
if backbone.conditioning_dim > 0:
    components["conditioning_encoder"] = backbone.conditioning_encoder
components["backbone_mlp"] = backbone.backbone
components["actor_head"] = policy.actor_head
components["critic_head"] = policy.critic_head

names, counts = zip(*[(k, count_params(v)) for k, v in components.items()])
total = sum(counts)

print(f"{'Component':>25s} | {'Params':>10s} | {'%':>6s}")
print("-" * 48)
for n, c in zip(names, counts):
    print(f"{n:>25s} | {c:>10,d} | {c / total:>5.1%}")
print("-" * 48)
print(f"{'TOTAL':>25s} | {total:>10,d}")
if SPLIT_NETWORK:
    critic_bb = count_params(policy.critic_backbone)
    print(f"{'+ critic_backbone':>25s} | {critic_bb:>10,d}")
    print(f"{'GRAND TOTAL':>25s} | {total + critic_bb:>10,d}")

fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.Set3(np.linspace(0, 1, len(names)))
bars = ax.barh(names, counts, color=colors, edgecolor="black")
for bar, c in zip(bars, counts):
    ax.text(bar.get_width() + total * 0.01, bar.get_y() + bar.get_height() / 2, f"{c:,}", va="center", fontsize=8)
ax.set_xlabel("Parameters")
ax.set_title(f"Parameter Distribution ({total:,} total)")
plt.tight_layout()
plt.show()

## Forward Pass Shape Trace

In [ ]:
x = obs_tensor
backbone = policy.actor_backbone

slide_idx = EGO_DIM
cond_dim = backbone.conditioning_dim
partner_dim = MAX_PARTNERS * PARTNER_F
lane_dim = MAX_LANES * ROAD_F
boundary_dim = MAX_BOUNDS * ROAD_F
traffic_dim = MAX_TRAFFIC * TRAFFIC_CONTROL_F

# Slicing
ego_obs = x[:, :slide_idx]
slices = [("ego", 0, slide_idx, ego_obs.shape)]

if cond_dim > 0:
    cond_obs = x[:, slide_idx : slide_idx + cond_dim]
    slices.append(("conditioning", slide_idx, slide_idx + cond_dim, cond_obs.shape))
    slide_idx += cond_dim

partner_obs = x[:, slide_idx : slide_idx + partner_dim]
slices.append(("partners", slide_idx, slide_idx + partner_dim, partner_obs.shape))
slide_idx += partner_dim

lane_obs = x[:, slide_idx : slide_idx + lane_dim]
slices.append(("lanes", slide_idx, slide_idx + lane_dim, lane_obs.shape))
slide_idx += lane_dim

boundary_obs = x[:, slide_idx : slide_idx + boundary_dim]
slices.append(("boundaries", slide_idx, slide_idx + boundary_dim, boundary_obs.shape))
slide_idx += boundary_dim

traffic_obs = x[:, slide_idx : slide_idx + traffic_dim]
slices.append(("traffic_ctrl", slide_idx, slide_idx + traffic_dim, traffic_obs.shape))

print(f"Obs buffer layout (total={x.shape[1]}):")
print(f"{'Name':>15s} | {'Start':>5s} | {'End':>5s} | {'Width':>5s} | Shape")
print("-" * 65)
for name, start, end, shape in slices:
    print(f"{name:>15s} | {start:>5d} | {end:>5d} | {end - start:>5d} | {shape}")

# Forward through encoders
print("\nEncoder outputs:")
with torch.no_grad():
    ego_enc = backbone.ego_encoder(ego_obs)
    print(f"  ego_encoder:     {ego_obs.shape} -> {ego_enc.shape}")

    if cond_dim > 0:
        cond_enc = backbone.conditioning_encoder(cond_obs)
        print(f"  cond_encoder:    {cond_obs.shape} -> {cond_enc.shape}")

    p_reshaped = partner_obs.view(-1, MAX_PARTNERS, PARTNER_F)
    p_enc, _ = backbone.partner_encoder(p_reshaped).max(dim=1)
    print(f"  partner_encoder: {partner_obs.shape} -> view {p_reshaped.shape} -> encode -> max-pool -> {p_enc.shape}")

    l_reshaped = lane_obs.view(-1, MAX_LANES, ROAD_F)
    l_enc, _ = backbone.lane_encoder(l_reshaped).max(dim=1)
    print(f"  lane_encoder:    {lane_obs.shape} -> view {l_reshaped.shape} -> encode -> max-pool -> {l_enc.shape}")

    b_reshaped = boundary_obs.view(-1, MAX_BOUNDS, ROAD_F)
    b_enc, _ = backbone.boundary_encoder(b_reshaped).max(dim=1)
    print(f"  bound_encoder:   {boundary_obs.shape} -> view {b_reshaped.shape} -> encode -> max-pool -> {b_enc.shape}")

    t_reshaped = traffic_obs.view(-1, MAX_TRAFFIC, TRAFFIC_CONTROL_F)
    t_cont = t_reshaped[:, :, : TRAFFIC_CONTROL_F - 2]
    t_type = t_reshaped[:, :, TRAFFIC_CONTROL_F - 2]
    t_state = t_reshaped[:, :, TRAFFIC_CONTROL_F - 1]
    t_type_onehot = F.one_hot(t_type.long(), num_classes=binding.NUM_TRAFFIC_CONTROL_TYPES).float()
    t_state_onehot = F.one_hot(t_state.long(), num_classes=binding.NUM_TRAFFIC_CONTROL_STATES).float()
    t_input = torch.cat([t_cont, t_type_onehot, t_state_onehot], dim=2)
    t_enc, _ = backbone.traffic_control_encoder(t_input).max(dim=1)
    print(
        f"  traffic_encoder: {traffic_obs.shape} -> view {t_reshaped.shape} -> onehot {t_input.shape} -> encode -> max-pool -> {t_enc.shape}"
    )

    # Concat + backbone
    features = [ego_enc, l_enc, b_enc, p_enc, t_enc]
    if cond_dim > 0:
        features.append(cond_enc)
    concat = torch.cat(features, dim=1)
    hidden = backbone.backbone(concat)
    print(f"\n  concat: {concat.shape}")
    print(f"  backbone_mlp: {concat.shape} -> {hidden.shape}")

    # Heads
    actor_out = policy.actor_head(hidden)
    critic_out = policy.critic_head(hidden)
    print(f"  actor_head:  {hidden.shape} -> {actor_out.shape} (split into {policy.atn_dim})")
    print(f"  critic_head: {hidden.shape} -> {critic_out.shape}")

## Weight Distributions by Layer

In [ ]:
weight_data = [
    (n, p.data.cpu().numpy().flatten()) for n, p in policy.named_parameters() if "weight" in n and p.dim() >= 2
]

n_weights = len(weight_data)
cols = 4
rows = (n_weights + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3 * rows))
axes = axes.flatten()

for i, (name, w) in enumerate(weight_data):
    ax = axes[i]
    ax.hist(w, bins=50, edgecolor="black", alpha=0.7, density=True)
    ax.set_title(name.replace("actor_backbone.", ""), fontsize=7)
    ax.axvline(0, color="red", ls="--", lw=0.5)
    ax.text(0.95, 0.95, f"std={w.std():.3f}", transform=ax.transAxes, fontsize=6, ha="right", va="top")

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Weight Distributions (init)", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Activation Analysis (per encoder)

In [ ]:
policy.eval()
with torch.no_grad():
    hidden = policy.actor_backbone(obs_tensor, EGO_DIM)
    action_logits, value = policy.decode_actions(hidden)

# Collect per-encoder activations
activations = {}
with torch.no_grad():
    slide = EGO_DIM
    activations["ego"] = backbone.ego_encoder(obs_tensor[:, :EGO_DIM])

    if cond_dim > 0:
        activations["conditioning"] = backbone.conditioning_encoder(obs_tensor[:, slide : slide + cond_dim])
        slide += cond_dim

    p_obs = obs_tensor[:, slide : slide + partner_dim].view(-1, MAX_PARTNERS, PARTNER_F)
    activations["partner"], _ = backbone.partner_encoder(p_obs).max(dim=1)
    slide += partner_dim

    l_obs = obs_tensor[:, slide : slide + lane_dim].view(-1, MAX_LANES, ROAD_F)
    activations["lane"], _ = backbone.lane_encoder(l_obs).max(dim=1)
    slide += lane_dim

    b_obs = obs_tensor[:, slide : slide + boundary_dim].view(-1, MAX_BOUNDS, ROAD_F)
    activations["boundary"], _ = backbone.boundary_encoder(b_obs).max(dim=1)
    slide += boundary_dim

    t_obs = obs_tensor[:, slide : slide + traffic_dim].view(-1, MAX_TRAFFIC, TRAFFIC_CONTROL_F)
    t_cont = t_obs[:, :, : TRAFFIC_CONTROL_F - 2]
    t_type = t_obs[:, :, TRAFFIC_CONTROL_F - 2]
    t_state = t_obs[:, :, TRAFFIC_CONTROL_F - 1]
    t_type_onehot = F.one_hot(t_type.long(), num_classes=binding.NUM_TRAFFIC_CONTROL_TYPES).float()
    t_state_onehot = F.one_hot(t_state.long(), num_classes=binding.NUM_TRAFFIC_CONTROL_STATES).float()
    t_input = torch.cat([t_cont, t_type_onehot, t_state_onehot], dim=2)
    activations["traffic_ctrl"], _ = backbone.traffic_control_encoder(t_input).max(dim=1)

    activations["hidden"] = hidden

fig, axes = plt.subplots(2, 4, figsize=(16, 6))
axes = axes.flatten()
for i, (name, act) in enumerate(activations.items()):
    if i >= len(axes):
        break
    vals = act.cpu().numpy().flatten()
    ax = axes[i]
    ax.hist(vals, bins=50, edgecolor="black", alpha=0.7)
    dead = (act.abs().sum(dim=0) == 0).sum().item()
    ax.set_title(f"{name} (dead={dead}/{act.shape[1]})", fontsize=9)
    ax.text(
        0.95,
        0.95,
        f"mean={vals.mean():.3f}\nstd={vals.std():.3f}",
        transform=ax.transAxes,
        fontsize=7,
        ha="right",
        va="top",
    )

for j in range(i + 1, len(axes)):
    axes[j].axis("off")

fig.suptitle("Per-Encoder Activation Distributions", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

## Encoder Embedding Similarity (cosine)

In [ ]:
# Mean embedding per encoder (exclude hidden — different dim)
emb_names = [k for k in activations.keys() if k != "hidden"]
emb_means = torch.stack([activations[k].mean(dim=0) for k in emb_names])
emb_norm = F.normalize(emb_means, dim=1)
sim_matrix = (emb_norm @ emb_norm.T).cpu().numpy()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(sim_matrix, cmap="RdBu_r", vmin=-1, vmax=1)
ax.set_xticks(range(len(emb_names)))
ax.set_yticks(range(len(emb_names)))
ax.set_xticklabels(emb_names, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(emb_names, fontsize=8)
for i in range(len(emb_names)):
    for j in range(len(emb_names)):
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center", fontsize=7)
fig.colorbar(im, ax=ax)
ax.set_title("Cosine Similarity Between Encoder Mean Embeddings")
plt.tight_layout()
plt.show()

## Architecture Comparison
Compare different architecture configs side-by-side without training.

In [ ]:
configs = [
    {"name": "tiny", "input_size": 32, "backbone_hidden_size": 64},
    {"name": "small", "input_size": 64, "backbone_hidden_size": 128},
    {"name": "medium", "input_size": 128, "backbone_hidden_size": 256, "backbone_num_layers": 2},
    {
        "name": "large",
        "input_size": 128,
        "backbone_hidden_size": 512,
        "backbone_num_layers": 2,
        "actor_num_layers": 2,
        "actor_hidden_size": 256,
        "critic_num_layers": 2,
        "critic_hidden_size": 256,
    },
    {
        "name": "xlarge",
        "input_size": 256,
        "backbone_hidden_size": 1024,
        "backbone_num_layers": 3,
        "actor_num_layers": 2,
        "actor_hidden_size": 512,
        "critic_num_layers": 2,
        "critic_hidden_size": 512,
    },
    {"name": "small+giga", "input_size": 64, "backbone_hidden_size": 128, "encoder_gigaflow": True, "dropout": 0.1},
    {
        "name": "medium+giga",
        "input_size": 128,
        "backbone_hidden_size": 256,
        "backbone_num_layers": 2,
        "encoder_gigaflow": True,
        "dropout": 0.1,
    },
]

POLICY_DEFAULTS = {
    "backbone_num_layers": 1,
    "actor_hidden_size": 128,
    "actor_num_layers": 0,
    "critic_hidden_size": 128,
    "critic_num_layers": 0,
    "encoder_gigaflow": False,
    "dropout": 0.0,
    "split_network": False,
}

results = []
for cfg in configs:
    name = cfg.pop("name")
    full_cfg = {**POLICY_DEFAULTS, **cfg}
    p = DrivePolicy(env, **full_cfg).to(device)
    n_params = sum(pp.numel() for pp in p.parameters())

    with torch.no_grad():
        import time

        t0 = time.time()
        for _ in range(100):
            p(obs_tensor)
        if device.type == "cuda":
            torch.cuda.synchronize()
        ms_per_fwd = (time.time() - t0) / 100 * 1000

    results.append({"name": name, "params": n_params, "ms/fwd": ms_per_fwd, **cfg})
    cfg["name"] = name  # restore
    del p

print(
    f"{'Config':>12s} | {'input':>5s} | {'bb_h':>5s} | {'bb_L':>4s} | {'act_h':>5s} | {'act_L':>5s} | {'crt_h':>5s} | {'crt_L':>5s} | {'giga':>5s} | {'Params':>10s} | {'ms/fwd':>8s}"
)
print("-" * 105)
for r in results:
    print(
        f"{r['name']:>12s} | {r['input_size']:>5d} | {r.get('backbone_hidden_size', 1024):>5d} | {r.get('backbone_num_layers', 1):>4d} | {r.get('actor_hidden_size', 1024):>5d} | {r.get('actor_num_layers', 1):>5d} | {r.get('critic_hidden_size', 1024):>5d} | {r.get('critic_num_layers', 1):>5d} | {str(r.get('encoder_gigaflow', False)):>5s} | {r['params']:>10,d} | {r['ms/fwd']:>7.2f}ms"
    )

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
names = [r["name"] for r in results]
params = [r["params"] for r in results]
times = [r["ms/fwd"] for r in results]

bar_colors = ["coral" if r.get("encoder_gigaflow") else "steelblue" for r in results]

axes[0].bar(names, params, color=bar_colors, edgecolor="black")
axes[0].set_ylabel("Parameters")
axes[0].set_title("Parameter Count (orange=gigaflow)")
axes[0].tick_params(axis="x", rotation=30)
for i, v in enumerate(params):
    axes[0].text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=7)

axes[1].bar(names, times, color=bar_colors, edgecolor="black")
axes[1].set_ylabel("ms / forward")
axes[1].set_title(f"Forward Pass Latency ({NUM_AGENTS} agents)")
axes[1].tick_params(axis="x", rotation=30)
for i, v in enumerate(times):
    axes[1].text(i, v, f"{v:.2f}", ha="center", va="bottom", fontsize=7)

plt.tight_layout()
plt.show()

## Observation Buffer Utilization
How much of each observation slot is actually filled (non-zero)?

In [ ]:
# Run a few steps to get diverse observations
actions = np.zeros((NUM_AGENTS, len(env.single_action_space.nvec)), dtype=np.int64)
all_obs = [obs]
for _ in range(20):
    o, _, _, _, _ = env.step(actions)
    all_obs.append(o)
stacked = np.concatenate(all_obs, axis=0)

slide = EGO_DIM
segments = [("ego", 0, EGO_DIM, 1, EGO_DIM)]
if cond_dim > 0:
    segments.append(("conditioning", slide, slide + cond_dim, 1, cond_dim))
    slide += cond_dim
segments.append(("partners", slide, slide + partner_dim, MAX_PARTNERS, PARTNER_F))
slide += partner_dim
segments.append(("lanes", slide, slide + lane_dim, MAX_LANES, ROAD_F))
slide += lane_dim
segments.append(("boundaries", slide, slide + boundary_dim, MAX_BOUNDS, ROAD_F))
slide += boundary_dim
segments.append(("traffic", slide, slide + traffic_dim, MAX_TRAFFIC, TRAFFIC_CONTROL_F))

print(f"{'Segment':>15s} | {'Slots':>5s} | {'Features':>8s} | {'Fill %':>7s} | {'Mean':>8s} | {'Std':>8s}")
print("-" * 65)
fill_rates = []
seg_names = []
for name, start, end, n_slots, n_feat in segments:
    chunk = stacked[:, start:end]
    if n_slots > 1:
        reshaped = chunk.reshape(-1, n_slots, n_feat)
        # A slot is "filled" if any feature is non-zero
        filled = (np.abs(reshaped).sum(axis=2) > 1e-8).mean()
    else:
        filled = (np.abs(chunk) > 1e-8).mean()
    fill_rates.append(filled * 100)
    seg_names.append(name)
    print(f"{name:>15s} | {n_slots:>5d} | {n_feat:>8d} | {filled:>6.1%} | {chunk.mean():>8.4f} | {chunk.std():>8.4f}")

fig, ax = plt.subplots(figsize=(8, 4))
colors = ["#2ecc71" if f > 50 else "#e74c3c" if f < 10 else "#f39c12" for f in fill_rates]
ax.barh(seg_names, fill_rates, color=colors, edgecolor="black")
ax.set_xlabel("Fill Rate (%)")
ax.set_title("Observation Slot Utilization")
ax.axvline(50, color="gray", ls="--", alpha=0.5)
for i, v in enumerate(fill_rates):
    ax.text(v + 1, i, f"{v:.1f}%", va="center", fontsize=8)
plt.tight_layout()
plt.show()

## LSTM Wrapper Architecture

In [ ]:
base_policy = DrivePolicy(
    env,
    input_size=INPUT_SIZE,
    backbone_hidden_size=BACKBONE_HIDDEN_SIZE,
    backbone_num_layers=BACKBONE_NUM_LAYERS,
    actor_hidden_size=ACTOR_HIDDEN_SIZE,
    actor_num_layers=ACTOR_NUM_LAYERS,
    critic_hidden_size=CRITIC_HIDDEN_SIZE,
    critic_num_layers=CRITIC_NUM_LAYERS,
    encoder_gigaflow=ENCODER_GIGAFLOW,
    dropout=DROPOUT,
    split_network=SPLIT_NETWORK,
).to(device)
# LSTM input_size must match backbone_hidden_size (backbone output dim)
lstm_policy = Recurrent(env, base_policy, input_size=BACKBONE_HIDDEN_SIZE, hidden_size=BACKBONE_HIDDEN_SIZE).to(device)

base_params = sum(p.numel() for p in base_policy.parameters())
lstm_params = sum(p.numel() for p in lstm_policy.parameters())
lstm_only = lstm_params - base_params

print(f"Base policy params:  {base_params:>10,d}")
print(f"LSTM wrapper params: {lstm_params:>10,d}")
print(f"LSTM overhead:       {lstm_only:>10,d} (+{lstm_only / base_params:.1%})")
print()

# torchinfo can't handle LSTMWrapper (requires state dict), so manual breakdown
print(f"{'Component':>25s} | {'Params':>10s}")
print("-" * 40)
for name, module in lstm_policy.named_children():
    n = sum(p.numel() for p in module.parameters())
    print(f"{name:>25s} | {n:>10,d}")
print()

# Verify forward pass works with proper state
# forward_eval uses LSTMCell which expects 2D state (batch, hidden)
with torch.no_grad():
    state = {
        "lstm_h": torch.zeros(NUM_AGENTS, BACKBONE_HIDDEN_SIZE, device=device),
        "lstm_c": torch.zeros(NUM_AGENTS, BACKBONE_HIDDEN_SIZE, device=device),
    }
    actions, value = lstm_policy.forward_eval(obs_tensor, state)
    if isinstance(actions, (list, tuple)):
        print(f"Forward eval OK: actions={[a.shape for a in actions]}, value={value.shape}")
    else:
        print(f"Forward eval OK: actions={actions}, value={value.shape}")

del base_policy, lstm_policy

## Effective Receptive Field
Which input features have the most influence on the hidden representation?

In [ ]:
# Jacobian-based sensitivity: d(hidden) / d(obs) magnitude
sample = obs_tensor[:1].clone().requires_grad_(True)
hidden = policy.actor_backbone(sample, EGO_DIM)
# Sum hidden to scalar for backward
hidden.sum().backward()
sensitivity = sample.grad.abs().squeeze().cpu().numpy()

fig, axes = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={"height_ratios": [2, 1]})

# Full sensitivity
axes[0].plot(sensitivity, lw=0.5, color="steelblue")
axes[0].set_ylabel("|grad|")
axes[0].set_title("Input Feature Sensitivity (|d hidden / d obs|)")

# Mark segments
seg_boundaries = [0, EGO_DIM]
seg_labels = ["ego"]
s = EGO_DIM
if cond_dim > 0:
    s += cond_dim
    seg_boundaries.append(s)
    seg_labels.append("cond")
for name, dim in [
    ("partners", partner_dim),
    ("lanes", lane_dim),
    ("boundaries", boundary_dim),
    ("traffic", traffic_dim),
]:
    s += dim
    seg_boundaries.append(s)
    seg_labels.append(name)

seg_colors = plt.cm.Set2(np.linspace(0, 1, len(seg_labels)))
for i, (label, c) in enumerate(zip(seg_labels, seg_colors)):
    start, end = seg_boundaries[i], seg_boundaries[i + 1]
    axes[0].axvspan(start, end, alpha=0.15, color=c)
    axes[0].text((start + end) / 2, axes[0].get_ylim()[1] * 0.9, label, ha="center", fontsize=7, color="black")

# Per-segment mean sensitivity
seg_means = []
for i in range(len(seg_labels)):
    start, end = seg_boundaries[i], seg_boundaries[i + 1]
    seg_means.append(sensitivity[start:end].mean())

axes[1].bar(seg_labels, seg_means, color=seg_colors, edgecolor="black")
axes[1].set_ylabel("Mean |grad|")
axes[1].set_title("Mean Sensitivity per Observation Segment")

plt.tight_layout()
plt.show()

policy.zero_grad()